# DL050 -- Deep Learning Prerequisites Quick Reference

---

## Learning Objectives

By the end of this notebook, you will:

- Review the **linear algebra** operations that underpin neural networks
- Build intuition for **calculus** concepts used in backpropagation
- Recall key **probability** ideas (distributions, Bayes' theorem)
- Refresh essential **Python** patterns (comprehensions, classes, decorators)
- Practice **NumPy**, **Matplotlib**, and **scikit-learn** fundamentals
- Know the most **common mistakes** beginners make and how to debug them

## Prerequisites

- Basic Python 3 (variables, loops, functions)
- Some exposure to linear algebra (vectors, matrices)

## Table of Contents

1. [Linear Algebra Refresher](#1.-Linear-Algebra-Refresher)
2. [Calculus Refresher](#2.-Calculus-Refresher)
3. [Probability Refresher](#3.-Probability-Refresher)
4. [Python Refresher](#4.-Python-Refresher)
5. [NumPy Essentials](#5.-NumPy-Essentials)
6. [Matplotlib Basics](#6.-Matplotlib-Basics)
7. [scikit-learn Basics](#7.-scikit-learn-Basics)
8. [Common Mistakes & Debugging Tips](#8.-Common-Mistakes-&-Debugging-Tips)

In [ ]:
# ── Global imports and reproducibility ──
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline

---

## 1. Linear Algebra Refresher

Almost every deep learning operation is a **matrix operation**. This section covers the essentials.

### 1.1 Vectors

A vector is an ordered list of numbers. In deep learning we think of them as points or directions in $n$-dimensional space.

$$\mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix} \in \mathbb{R}^n$$

In [ ]:
# Vectors in NumPy
x = np.array([1.0, 2.0, 3.0])
y = np.array([4.0, 5.0, 6.0])

print("x          :", x)
print("y          :", y)
print("x + y      :", x + y)       # element-wise addition
print("3 * x      :", 3 * x)       # scalar multiplication
print("||x|| (L2) :", np.linalg.norm(x))  # Euclidean norm

### 1.2 Dot Product

The dot product measures how "aligned" two vectors are.

$$\mathbf{x} \cdot \mathbf{y} = \sum_{i=1}^{n} x_i y_i$$

- If $\mathbf{x} \cdot \mathbf{y} > 0$: vectors point in roughly the same direction
- If $\mathbf{x} \cdot \mathbf{y} = 0$: vectors are orthogonal (perpendicular)
- If $\mathbf{x} \cdot \mathbf{y} < 0$: vectors point in roughly opposite directions

In [ ]:
dot_val = np.dot(x, y)
print(f"x . y = {dot_val}")  # 1*4 + 2*5 + 3*6 = 32

# Equivalently:
print(f"sum(x*y) = {np.sum(x * y)}")
print(f"x @ y    = {x @ y}")

### 1.3 Matrices

A matrix is a 2D array of numbers. In deep learning, weight matrices store the learnable parameters.

$$\mathbf{W} \in \mathbb{R}^{m \times n} = \begin{bmatrix} w_{11} & w_{12} & \cdots & w_{1n} \\ w_{21} & w_{22} & \cdots & w_{2n} \\ \vdots & & \ddots & \vdots \\ w_{m1} & w_{m2} & \cdots & w_{mn} \end{bmatrix}$$

In [ ]:
W = np.array([[1, 2],
              [3, 4],
              [5, 6]])

print("W:\n", W)
print("Shape :", W.shape)       # (3, 2) -- 3 rows, 2 columns
print("W^T:\n", W.T)           # transpose: (2, 3)
print("Element [1,0]:", W[1, 0])  # row 1, col 0 -> 3

### 1.4 Matrix Multiplication

The core operation in neural networks: $\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}$

For $\mathbf{A} \in \mathbb{R}^{m \times n}$ and $\mathbf{B} \in \mathbb{R}^{n \times p}$, the result $\mathbf{C} = \mathbf{A}\mathbf{B} \in \mathbb{R}^{m \times p}$ where:

$$C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}$$

**Key rule:** Inner dimensions must match: $(m \times \mathbf{n}) \cdot (\mathbf{n} \times p) \rightarrow (m \times p)$

In [ ]:
A = np.random.randn(3, 4)  # 3x4
B = np.random.randn(4, 2)  # 4x2

C = A @ B  # 3x2
print("A shape:", A.shape)
print("B shape:", B.shape)
print("C = A @ B shape:", C.shape)
print("C:\n", C.round(3))

# Neural network forward pass example
W = np.random.randn(3, 5)      # weights: 3 outputs, 5 inputs
x = np.random.randn(5, 1)      # input: 5 features, 1 sample
b = np.random.randn(3, 1)      # bias: 3 outputs

z = W @ x + b
print("\nForward pass z = Wx + b:\n", z.round(3))

---

## 2. Calculus Refresher

Deep learning learns by computing **gradients** (derivatives) and using them to update weights. You do not need to be a calculus expert -- but you need intuition for what a derivative means.

### 2.1 Derivatives

The derivative $f'(x) = \frac{df}{dx}$ tells you **how fast** $f$ changes when you nudge $x$.

$$f'(x) = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}$$

**Common derivatives:**

| $f(x)$ | $f'(x)$ |
|---------|----------|
| $x^n$ | $n x^{n-1}$ |
| $e^x$ | $e^x$ |
| $\ln(x)$ | $1/x$ |
| $\sigma(x) = \frac{1}{1+e^{-x}}$ | $\sigma(x)(1 - \sigma(x))$ |

In [ ]:
def numerical_derivative(f, x, h=1e-7):
    """Approximate the derivative of f at x using central differences."""
    return (f(x + h) - f(x - h)) / (2 * h)

# Example: f(x) = x^2, f'(x) = 2x
f = lambda x: x**2

x_val = 3.0
approx = numerical_derivative(f, x_val)
exact = 2 * x_val

print(f"f(x) = x^2 at x = {x_val}")
print(f"  Numerical derivative: {approx:.6f}")
print(f"  Exact derivative:     {exact:.6f}")

In [ ]:
# Visualize: the derivative is the slope of the tangent line
x = np.linspace(-3, 3, 200)
f_vals = x**2
f_prime = 2 * x  # analytical derivative

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(x, f_vals, label="$f(x) = x^2$")
axes[0].set_title("Function")
axes[0].set_xlabel("x")
axes[0].set_ylabel("f(x)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, f_prime, label="$f'(x) = 2x$", color="orange")
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].set_title("Derivative (slope)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("f'(x)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.2 The Chain Rule

The chain rule is the mathematical engine behind **backpropagation**.

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx} = f'(g(x)) \cdot g'(x)$$

**Example:** $y = (3x + 1)^2$
- Let $g(x) = 3x + 1$, $f(g) = g^2$
- $\frac{dy}{dx} = 2g \cdot 3 = 6(3x + 1)$

In [ ]:
# Verify chain rule numerically
def composed(x):
    return (3 * x + 1) ** 2

def analytical_deriv(x):
    return 6 * (3 * x + 1)

x_test = 2.0
print(f"Numerical : {numerical_derivative(composed, x_test):.6f}")
print(f"Analytical: {analytical_deriv(x_test):.6f}")

### 2.3 Partial Derivatives

When a function has multiple inputs, we take the derivative with respect to **one variable at a time**, treating the others as constants.

For $f(x, y) = x^2 y + 3y$:

$$\frac{\partial f}{\partial x} = 2xy, \qquad \frac{\partial f}{\partial y} = x^2 + 3$$

The **gradient** collects all partial derivatives into a vector:

$$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix}$$

The gradient points in the direction of **steepest ascent**. Gradient descent goes the opposite way.

In [ ]:
def f(x, y):
    return x**2 * y + 3 * y

def grad_f(x, y):
    """Analytical gradient of f."""
    df_dx = 2 * x * y
    df_dy = x**2 + 3
    return np.array([df_dx, df_dy])

# Numerical check using finite differences
x0, y0 = 2.0, 3.0
h = 1e-7

df_dx_num = (f(x0 + h, y0) - f(x0 - h, y0)) / (2 * h)
df_dy_num = (f(x0, y0 + h) - f(x0, y0 - h)) / (2 * h)

print(f"Analytical gradient at ({x0}, {y0}): {grad_f(x0, y0)}")
print(f"Numerical  gradient at ({x0}, {y0}): [{df_dx_num:.4f}, {df_dy_num:.4f}]")

---

## 3. Probability Refresher

Deep learning uses probability for:
- **Loss functions** (cross-entropy)
- **Output layers** (softmax produces a probability distribution)
- **Regularization** (dropout samples from Bernoulli)
- **Generative models** (VAEs, diffusion models)

### 3.1 Key Distributions

| Distribution | Use in DL | Parameters |
|-------------|-----------|------------|
| **Bernoulli** | Dropout, binary classification | $p$ (probability of 1) |
| **Gaussian (Normal)** | Weight initialization, noise | $\mu$ (mean), $\sigma^2$ (variance) |
| **Categorical** | Multi-class classification (softmax output) | $p_1, \ldots, p_K$ |
| **Uniform** | Random initialization | $a$ (min), $b$ (max) |

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

# Gaussian
samples = np.random.randn(10000)
axes[0].hist(samples, bins=60, density=True, alpha=0.7, color="steelblue")
axes[0].set_title("Gaussian ($\\mu=0, \\sigma=1$)")
axes[0].set_xlabel("x")

# Uniform
samples_u = np.random.uniform(-1, 1, 10000)
axes[1].hist(samples_u, bins=60, density=True, alpha=0.7, color="coral")
axes[1].set_title("Uniform ($a=-1, b=1$)")
axes[1].set_xlabel("x")

# Bernoulli (simulated as binary)
p = 0.3
samples_b = np.random.binomial(1, p, 10000)
axes[2].bar([0, 1], [np.mean(samples_b == 0), np.mean(samples_b == 1)],
            color=["gray", "green"], alpha=0.7)
axes[2].set_title(f"Bernoulli ($p={p}$)")
axes[2].set_xticks([0, 1])
axes[2].set_xlabel("x")
axes[2].set_ylabel("proportion")

plt.tight_layout()
plt.show()

### 3.2 Bayes' Theorem

$$P(A \mid B) = \frac{P(B \mid A) \, P(A)}{P(B)}$$

- $P(A \mid B)$: **posterior** -- updated belief about $A$ after observing $B$
- $P(B \mid A)$: **likelihood** -- how likely is $B$ if $A$ is true
- $P(A)$: **prior** -- initial belief about $A$
- $P(B)$: **evidence** -- total probability of observing $B$

**Quick example:** A medical test is 99% accurate. The disease affects 1 in 1000 people. If you test positive, what is the probability you actually have the disease?

In [ ]:
# Bayes' theorem in action
P_disease = 0.001           # prior: 1 in 1000
P_positive_given_disease = 0.99   # sensitivity (true positive rate)
P_positive_given_healthy = 0.01   # false positive rate

# P(positive) = P(pos|disease)*P(disease) + P(pos|healthy)*P(healthy)
P_positive = (P_positive_given_disease * P_disease +
              P_positive_given_healthy * (1 - P_disease))

# Posterior: P(disease | positive)
P_disease_given_positive = (P_positive_given_disease * P_disease) / P_positive

print(f"P(disease | positive test) = {P_disease_given_positive:.4f}")
print(f"That is only {P_disease_given_positive*100:.1f}% -- most positives are false positives!")

---

## 4. Python Refresher

### 4.1 List Comprehensions

Compact syntax for creating lists. Used everywhere in data processing.

In [ ]:
# Basic list comprehension
squares = [x**2 for x in range(10)]
print("Squares:", squares)

# Filtered
even_squares = [x**2 for x in range(10) if x % 2 == 0]
print("Even sq:", even_squares)

# Nested
pairs = [(i, j) for i in range(3) for j in range(3) if i != j]
print("Pairs  :", pairs)

# Dict comprehension
word_lengths = {w: len(w) for w in ["relu", "sigmoid", "tanh", "softmax"]}
print("Lengths:", word_lengths)

### 4.2 Classes

PyTorch models are Python classes. You need to be comfortable with `__init__`, `self`, and inheritance.

In [ ]:
class Neuron:
    """A single neuron with sigmoid activation."""

    def __init__(self, n_inputs):
        np.random.seed(42)
        self.weights = np.random.randn(n_inputs)
        self.bias = 0.0

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))

    def forward(self, x):
        z = np.dot(self.weights, x) + self.bias
        return self.sigmoid(z)

    def __repr__(self):
        return f"Neuron(weights={self.weights.round(3)}, bias={self.bias})"


neuron = Neuron(n_inputs=3)
print(neuron)

x_input = np.array([1.0, 0.5, -1.0])
output = neuron.forward(x_input)
print(f"Output for {x_input}: {output:.4f}")

### 4.3 Decorators

Decorators wrap a function to add behavior. You will see them in PyTorch (`@torch.no_grad()`, `@staticmethod`).

In [ ]:
import time

def timer(func):
    """Decorator that prints execution time."""
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper


@timer
def slow_dot(a, b):
    """Dot product using a Python loop (slow)."""
    total = 0.0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total


@timer
def fast_dot(a, b):
    """Dot product using NumPy (fast)."""
    return np.dot(a, b)


a = np.random.randn(1_000_000)
b = np.random.randn(1_000_000)

_ = slow_dot(a, b)
_ = fast_dot(a, b)
print("Lesson: always use vectorized NumPy operations!")

---

## 5. NumPy Essentials

### 5.1 Array Creation

In [ ]:
np.random.seed(42)

# From list
a = np.array([1, 2, 3])

# Common constructors
zeros = np.zeros((2, 3))
ones  = np.ones((2, 3))
eye   = np.eye(3)          # 3x3 identity
rng   = np.arange(0, 10, 2)
lin   = np.linspace(0, 1, 5)

# Random
uniform = np.random.rand(2, 3)      # U(0,1)
normal  = np.random.randn(2, 3)     # N(0,1)
ints    = np.random.randint(0, 10, size=(2, 3))

print("zeros:\n", zeros)
print("eye:\n", eye)
print("linspace:", lin)
print("normal:\n", normal.round(3))

### 5.2 Broadcasting

Broadcasting lets NumPy perform operations on arrays with different shapes. Rules:

1. Dimensions are compared from the **right**
2. Dimensions match if they are equal, or one of them is **1**
3. Missing dimensions are treated as 1

In [ ]:
M = np.arange(12).reshape(3, 4)   # shape (3, 4)
row = np.array([10, 20, 30, 40])  # shape (4,) -> broadcast to (3, 4)
col = np.array([[100], [200], [300]])  # shape (3, 1) -> broadcast to (3, 4)

print("M:\n", M)
print("\nM + row:\n", M + row)
print("\nM + col:\n", M + col)

# Common pattern: subtract mean from each column
col_means = M.mean(axis=0)   # shape (4,)
centered = M - col_means
print("\nColumn means:", col_means)
print("Centered:\n", centered)

### 5.3 Reshaping

In [ ]:
a = np.arange(24)

print("Original shape:", a.shape)           # (24,)
print("Reshaped (4,6):\n", a.reshape(4, 6))
print("Reshaped (2,3,4):\n", a.reshape(2, 3, 4))
print("Using -1 for auto:\n", a.reshape(6, -1))  # infers 4

# Flatten: multi-dim -> 1D
b = a.reshape(2, 3, 4)
print("\nFlattened:", b.flatten().shape)   # (24,)
print("Ravel:   ", b.ravel().shape)       # same but may share memory

### 5.4 Random Seed for Reproducibility

Always set a seed so experiments are repeatable.

In [ ]:
# Legacy API (simple, widely used)
np.random.seed(42)
print("Legacy:", np.random.randn(3).round(4))

np.random.seed(42)
print("Same  :", np.random.randn(3).round(4))  # identical

# Modern API (recommended for new code)
rng = np.random.default_rng(42)
print("\nModern:", rng.standard_normal(3).round(4))

rng2 = np.random.default_rng(42)
print("Same  :", rng2.standard_normal(3).round(4))

---

## 6. Matplotlib Basics

### 6.1 Line Plots

In [ ]:
x = np.linspace(-5, 5, 200)

# Common activation functions
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)

plt.figure(figsize=(8, 4))
plt.plot(x, sigmoid, label="Sigmoid", linewidth=2)
plt.plot(x, tanh, label="Tanh", linewidth=2)
plt.plot(x, relu, label="ReLU", linewidth=2)
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.title("Common Activation Functions")
plt.xlabel("z")
plt.ylabel("a(z)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2 Scatter Plots

In [ ]:
np.random.seed(42)

# Generate two clusters
n = 100
class_0 = np.random.randn(n, 2) + np.array([2, 2])
class_1 = np.random.randn(n, 2) + np.array([-2, -2])

plt.figure(figsize=(5, 5))
plt.scatter(class_0[:, 0], class_0[:, 1], label="Class 0", alpha=0.6, s=20)
plt.scatter(class_1[:, 0], class_1[:, 1], label="Class 1", alpha=0.6, s=20)
plt.title("Two-Class Scatter Plot")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis("equal")
plt.tight_layout()
plt.show()

### 6.3 Subplots

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(2, 2, figsize=(8, 6))

# Top-left: line
t = np.linspace(0, 4 * np.pi, 200)
axes[0, 0].plot(t, np.sin(t), color="steelblue")
axes[0, 0].set_title("Sine wave")

# Top-right: histogram
axes[0, 1].hist(np.random.randn(1000), bins=30, color="coral", alpha=0.7)
axes[0, 1].set_title("Gaussian histogram")

# Bottom-left: bar
categories = ["ReLU", "Sigmoid", "Tanh", "GELU"]
values = [45, 25, 15, 15]
axes[1, 0].bar(categories, values, color="mediumseagreen", alpha=0.7)
axes[1, 0].set_title("Activation popularity")

# Bottom-right: scatter
axes[1, 1].scatter(np.random.randn(50), np.random.randn(50),
                   c=np.random.randn(50), cmap="coolwarm", alpha=0.7)
axes[1, 1].set_title("Random scatter")

plt.tight_layout()
plt.show()

---

## 7. scikit-learn Basics

We use scikit-learn mainly for **data splitting**, **preprocessing**, and **evaluation metrics**.

### 7.1 Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

# Create a synthetic dataset
X, y = make_classification(
    n_samples=200,
    n_features=4,
    n_informative=2,
    n_redundant=1,
    random_state=42
)

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Full dataset : {X.shape[0]} samples, {X.shape[1]} features")
print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"Class balance (train): {np.bincount(y_train)}")
print(f"Class balance (test) : {np.bincount(y_test)}")

### 7.2 StandardScaler

Standardization transforms features to have **zero mean** and **unit variance**: $x' = \frac{x - \mu}{\sigma}$

This is critical for neural networks -- unscaled features cause slow or unstable training.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on training data ONLY, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # uses train statistics!

print("Before scaling:")
print(f"  Train mean: {X_train.mean(axis=0).round(3)}")
print(f"  Train std : {X_train.std(axis=0).round(3)}")

print("\nAfter scaling:")
print(f"  Train mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"  Train std : {X_train_scaled.std(axis=0).round(3)}")

### 7.3 Evaluation Metrics

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Train a simple model
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

---

## 8. Common Mistakes & Debugging Tips

### Shape Mismatches

The single most common error in deep learning code.

- **Symptom**: `ValueError: shapes (X,Y) and (Z,W) not aligned`
- **Fix**: Print `.shape` at every step. Remember: `(m, n) @ (n, p) -> (m, p)`
- **Tip**: Use `assert` statements liberally:

In [ ]:
W = np.random.randn(3, 5)
x = np.random.randn(5, 1)

# Good practice: assert shapes before operations
assert W.shape[1] == x.shape[0], f"Shape mismatch: W is {W.shape}, x is {x.shape}"
y = W @ x
assert y.shape == (W.shape[0], x.shape[1]), f"Unexpected output shape: {y.shape}"

print(f"W {W.shape} @ x {x.shape} = y {y.shape}  -- OK")

### Data Leakage

Fitting the scaler (or any preprocessing) on the **full dataset** before splitting leaks test information into training.

```python
# WRONG -- leaks test info into scaler statistics
scaler.fit(X)            # fits on ALL data
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled)

# RIGHT -- fit only on training data
X_train, X_test = train_test_split(X)
scaler.fit(X_train)      # fits on train only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
```

### Forgetting to Set Seeds

- Random initialization means different results each run
- Always set `np.random.seed(42)` (and `torch.manual_seed(42)` for PyTorch) at the top of your notebook

### Numerical Instability

- **Overflow**: $e^{1000} = \text{inf}$ -- use the log-sum-exp trick
- **Underflow**: $\log(0) = -\text{inf}$ -- add a small epsilon: `np.log(p + 1e-8)`
- **Vanishing gradients**: Very deep networks with sigmoid activations -- use ReLU or batch normalization

In [ ]:
# Numerical stability examples

# BAD: naive softmax overflows for large inputs
def softmax_naive(z):
    return np.exp(z) / np.sum(np.exp(z))

# GOOD: subtract max for numerical stability
def softmax_stable(z):
    z_shifted = z - np.max(z)  # prevents overflow
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z)

z = np.array([1000, 1001, 1002])

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    print("Naive softmax  :", softmax_naive(z))   # [nan, nan, nan]

print("Stable softmax :", softmax_stable(z))     # correct result

In [ ]:
# Safe log: avoid log(0)
def safe_log(x, eps=1e-8):
    return np.log(x + eps)

print("log(0)           :", np.log(0))        # -inf (warning)
print("safe_log(0)      :", safe_log(0))      # finite number
print("safe_log(1)      :", safe_log(1))      # ~0

### Debugging Checklist

When something goes wrong, check these in order:

1. **Shapes**: Print `.shape` at every major step
2. **Data types**: Check `.dtype` -- mixing `float32` and `float64` can cause issues
3. **NaN / Inf**: Use `np.isnan(x).any()` and `np.isinf(x).any()`
4. **Learning rate**: Too high causes divergence, too low causes no progress
5. **Data**: Visualize a few samples -- is the input sensible?
6. **Labels**: Are labels correctly aligned with inputs? Check `y[:5]` vs `X[:5]`
7. **Overfitting on one batch**: If the model cannot memorize one batch, there is a bug in the model

---

**You are now ready for DL100 -- Neural Network Fundamentals.** If any section above was unclear, revisit it before moving on.